# 📑 EOM - Outbound Calls Report
 
## 🔗 Links
- **[BRD Outbound Calls - TBD](tbd)**  
  Comprehensive documentation outlining the business requirements for this report.
 
- **[Story Ticket - TBD](tbd)**  
  Details about the story ticket, including tasks and progress.
  
- **[UAT Findings Ticket]()**  
  A summary of findings from UAT.
 
---

In [0]:
# %pip install openpyxl
# %restart_python

In [0]:
import os
import sys

dbutils.widgets.text("run_date", "")
dbutils.widgets.text("adhoc_run", '')

try:
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    repo_root = "/Workspace" + notebook_path.rsplit("/HoistInvestorReports/", 1)[0] 
    sys.path.insert(0, os.path.join(repo_root, "aws_utils"))
    sys.path.insert(0, os.path.join(repo_root, "01-utils"))
except Exception:
    sys.path.append("../../aws_utils")
    sys.path.append("../../01-utils")

from aws_utils import get_env_prefix, get_env_fullname
from helpers import load_configs, build_portfolio_filter

env = get_env_prefix()


In [0]:
from datetime import datetime
from pytz import timezone
from pyspark.sql import functions as F
from pyspark.sql.window import Window

athens_tz = timezone('Europe/Athens')
adhoc_run = dbutils.widgets.get("adhoc_run")

if adhoc_run == '1':
    dbutils.widgets.text("run_date", '')
    run_date = dbutils.widgets.get("run_date")
else:
    now_date = datetime.now(athens_tz).strftime('%Y-%m-%d')
    now_date_str = datetime.now(athens_tz).strftime('%Y%m%d')
    run_date = spark.sql(f"SELECT CurrentMonthLastWorkingDate FROM `{env}-cdh`.reference.ref_calendar WHERE CalendarDate = last_day(add_months(to_date('{now_date}'), -1))").first()["CurrentMonthLastWorkingDate"]

now_datetime_str = datetime.now(athens_tz).strftime('%Y%m%dT%H%M%S')

In [0]:
from typing import Any
from delta import DeltaTable  # type: ignore
from pyspark.sql.functions import (
    coalesce,
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    trim,
    when,
    months_between,
    dateadd,
    datediff,
    regexp_replace
)

from typing import Optional
import delta  # type: ignore
from pyspark.sql import SparkSession
import pandas as pd

In [0]:
## Portfolios In Scope - All HOIST portfolios
PortfolioName = "('Cell', 'Orbit', 'Pearl Portfolio A', 'Mercury', 'Light')"    

DecomissionDate = "2026-06-04"

##### Views

In [0]:
calendar = spark.sql(f"""
          SELECT *
          FROM `{env}-cdh`.reference.ref_calendar
      """)
calendar.createOrReplaceTempView("calendar")

dialer = spark.sql(f"""
SELECT *
from `{env}-cdh`.primary.dialer
where PortfolioName in {PortfolioName}                      
and businessdate = (SELECT max(businessdate)
                    from `{env}-cdh`.primary.dialer  ) 
AND LAST_DAY(CAST(DateHour AS DATE))=LAST_DAY('{run_date}')
""")
dialer.createOrReplaceTempView("dialer")   

In [0]:
reassignments = spark.sql(f"""                             
        select distinct re.LoanID
        from `{env}-reportinputfiles`.investor.hoist_reassignments as re
          """)
reassignments.createOrReplaceTempView("reassignments")

In [0]:
dexposure = spark.sql(
f'''
SELECT DISTINCT ExposureKey
FROM `{env}-cdh`.derivative.get_exposures_by_business_date('{run_date}', true)  dexp
LEFT JOIN reassignments as re on re.LoanID = dexp.ExposureKey
WHERE RE.LoanID IS NULL AND (dexp.JUSTIFICATIONCODE IS NULL OR dexp.JUSTIFICATIONCODE NOT IN (43055,43056))
'''
)
dexposure.createOrReplaceTempView("dexposure")

In [0]:
interaction = spark.sql(
f'''
SELECT DISTINCT
CASE WHEN LEFT(intr.ExposureKey,3) = md.Prefix THEN intr.ExposureKey ELSE CONCAT(md.Prefix,intr.ExposureKey) END ExposureKey,
intr.PortfolioName,
CASE WHEN LEFT(intr.CDI,3) = md.Prefix THEN intr.CDI ELSE CONCAT(md.Prefix,intr.CDI) END CDI,
INTR.InteractionTypeCode,
INTR.INTERACTIONTYPE,
CAST(INTR.InteractionDate AS DATE) AS InteractionDate,
INTR.InteractionKey,
INTR.INTERACTIONTYPEKEY,
intr.InteractionCategory,
intr.FirstLevelCategory,
intr.SecondLevelCategory,
intr.AttemptOutcome,
intr.IsPositiveFlag,
intr.IsPromiseFlag,
intr.ContactType,
intr.IsCallFlag,
intr.AttemptType,
intr.IsFtFAppointmentFlag,
intr.createddate,
intr.PortfolioName, 
intr.SourceSystem,
INTR.ContactTypeCode,
intr.ExternalCompanyKey,
intr.CreatedBy
from 
(
SELECT * FROM `{env}-cdh`.primary.get_interactions_by_business_date('{DecomissionDate}') intr
WHERE   CAST(interactiondate AS DATE) <= '{run_date}'
AND cast(interactiondate as date) >= trunc('{run_date}', 'MM')
AND InteractionDate <= '{DecomissionDate}'
and intr.PortfolioName in {PortfolioName}

UNION ALL 

SELECT * FROM `{env}-cdh`.primary.get_interactions_by_business_date('{run_date}') intr
WHERE   cast(interactiondate as date) <= '{run_date}'
AND cast(interactiondate as date) >= trunc('{run_date}', 'MM')
AND InteractionDate > '{DecomissionDate}' 
and intr.PortfolioName in {PortfolioName}
) intr 
left join `{env}-cdh`.primary.portfolio md 
on md.PortfolioName = intr.PortfolioName
and md.BusinessDate = '{run_date}'
'''
)
interaction.createOrReplaceTempView("interaction")

In [0]:
Talktime = spark.sql(f"""
		SELECT
			   cast(DL.DateHour AS DATE) AS DateHour
			  ,DL.PortfolioName
			  ,AVG(dl.TalkTimeDuration )          AS TalkTimeDuration 
		FROM dialer dl 		
  		WHERE 1 = 1
	  	AND dl.CampaignType = 'OUT'
	  	AND dl.CallDisposition IN ('Answered','ANSWER') 
	  	AND cast(dl.TalkTimeDuration AS INT)>0
	  	AND dl.TalkTimeDuration<>''
		GROUP BY cast(DL.DateHour AS DATE), DL.PortfolioName

""")
Talktime.createOrReplaceTempView("Talktime")   

In [0]:
DialerActivities = spark.sql(f"""
		SELECT
			 	cast(R.DateHour AS DATE) AS DateHour
			  , R.PortfolioName
			  , count(R.DateHour) 						      AS DialerUnsuccessfulAttempts
		FROM dialer  r 
		WHERE 1 = 1
	    AND r.DateHour >= '2019-01-01' 
		AND r.CallDisposition IN ('BUSY', 'CANCEL', 'ABANDONED', 'NO ANSWER')
		AND r.CDI IS NOT NULL
		AND ((r.CampaignType = 'OUT' AND r.IsExternalFlag=false) OR (r.IsExternalFlag=true))
		GROUP BY cast(R.DateHour AS DATE) , R.PortfolioName

""")
DialerActivities.createOrReplaceTempView("DialerActivities")   

In [0]:
InteractionCategory = spark.sql(f"""
	SELECT
      intr.PortfolioName,
			intr.ExposureKey,
      intr.CDI,
      intr.InteractionKey,
      intr.InteractionTypeKey, 
      intr.InteractionType,
      CAST(intr.InteractionDate as date) InteractionDate,
      intr.ContactType,
      INTR.ContactTypeCode,
      intr.InteractionCategory,
      intr.SecondLevelCategory,
      intr.AttemptType,
      intr.AttemptOutcome,
      intr.isPositiveFlag,
      intr.isPromiseFlag,
      intr.IsCallFlag,
      intr.IsFtFAppointmentFlag,
      intr.CreatedDate,
      intr.ExternalCompanyKey,
      intr.CreatedBy,
		  CASE 
          WHEN intr.InteractionDate <= '{DecomissionDate}'  and intr.InteractionType ILIKE '%refuse to pay%' AND intr.InteractionTypeKey not in ('1020', '3086') THEN 1 
          WHEN intr.InteractionDate > '{DecomissionDate}'  and (intr.InteractionType ILIKE '%refuse to pay%' or intr.InteractionType ILIKE '%unable to pay%' or intr.InteractionType ILIKE '%rtp%' ) AND intr.InteractionTypeKey not in ('8693', '7020') THEN 1 
          else 0 end as negative ,
      case 
          WHEN intr.InteractionDate <= '{DecomissionDate}'  and intr.InteractionType ILIKE '%promise to pay%' and intr.InteractionTypeKey not in ('1020', '3086') THEN  1 
          WHEN intr.InteractionDate > '{DecomissionDate}'  and (intr.InteractionType ILIKE '%promise to pay%' or intr.InteractionType ILIKE '%promise for down pay%')  and intr.InteractionTypeKey not in ('8693', '7020') THEN  1
          else 0 end as positive,
		  CASE 
          WHEN intr.InteractionDate <= '{DecomissionDate}'  and (upper(COALESCE(intr.ContactType, '')) IN ('RIGHT PARTY CONTACT', 'FACE TO FACE', 'RIGHT PERSON CONTACT') 
  				OR LOWER(COALESCE(intr.InteractionType, '')) LIKE '%planning business trip%') and intr.InteractionTypeKey not in ('1020', '3086')  THEN 1 
          WHEN CAST(intr.InteractionDate AS DATE) > '{DecomissionDate}' AND (intr.FirstLevelCategory NOT IN ('1. no contact', '2. contact - Not RPC')) and intr.InteractionTypeKey not in ('8693', '7020')  THEN 1 
          ELSE 0 END AS RPC,
      CASE 
          WHEN intr.InteractionDate <= '{DecomissionDate}' and (intr.ContactType IN ('Right Person Contact', '3rd Person Contact', 'RIGHT PARTY CONTACT','Dialer Answer') OR LOWER(COALESCE(intr.InteractionType, '')) LIKE '%planning business trip%' OR intr.InteractionTypeKey = '1010') and intr.InteractionTypeKey not in ('1020', '3086')  THEN 1 
          WHEN CAST(intr.InteractionDate AS DATE) > '{DecomissionDate}' AND (intr.FirstLevelCategory <> '1. no contact' or intr.ContactType = 'Dialer Answer') and intr.InteractionTypeKey not in ('8693', '7020')  THEN 1 
          ELSE 0 END AS Contact,
      1 AS Attempt,
        CASE WHEN intr.InteractionDate <= '{DecomissionDate}' and intr.InteractionTypeKey = '3028'  THEN 'Offer' 
             WHEN intr.InteractionDate <= '{DecomissionDate}' and intr.InteractionTypeKey = '3005'  THEN 'Approved' 
             WHEN intr.InteractionDate > '{DecomissionDate}' and intr.InteractionTypeKey = '8712'  THEN 'Offer' 
             WHEN intr.InteractionDate > '{DecomissionDate}' and intr.InteractionTypeKey = '8701'  THEN 'Approved' 
        END AS SettlementActivity
	
	FROM  interaction as intr
  LEFT JOIN dexposure de on intr.ExposureKey = de.ExposureKey 
  WHERE 1=1 
            AND    UPPER(COALESCE(ContactType, '')) <> 'SMS'

            AND (
                intr.ContactType IN ('Right Person Contact', '3rd Person Contact','RIGHT PARTY CONTACT') 
                OR COALESCE(intr.IsCallFlag, false) = true
                OR COALESCE(intr.IsFtFAppointmentFlag, false) = true
                OR LOWER(COALESCE(intr.InteractionType, '')) LIKE '%appointment%'
                OR LOWER(COALESCE(intr.AttemptType, '')) LIKE '%unsuccess%'
                OR LOWER(COALESCE(intr.AttemptOutcome, '')) LIKE '%no answer%'
                OR LOWER(COALESCE(intr.AttemptOutcome, '')) LIKE '%wrong%'
                OR LOWER(COALESCE(intr.InteractionType, '')) LIKE '%planning business trip%'
                OR 
                (FirstLevelCategory <> '0. Legal Action - no contact' 
                AND NOT (FirstLevelCategory = "1. no contact" AND SecondLevelCategory = "1.2 no contact - attempt")
                )
                )

            AND not (intr.InteractionTypeKey in ('1020', '3086', '991') and intr.InteractionDate <= '{DecomissionDate}' or intr.InteractionTypeKey in ('8693', '7020', '8795') and intr.InteractionDate > '{DecomissionDate}' )
            AND de.ExposureKey IS NOT NULL 
            

""")
InteractionCategory.createOrReplaceTempView("InteractionCategory")   

In [0]:
interactionsAggregated = spark.sql(f"""
	SELECT
			intr.PortfolioName,
            intr.Interactiondate, 
            intr.CDI, 
            intr.Exposurekey,
            COUNT(distinct CASE WHEN intr.Attempt  = 1 THEN intr.InteractionKey ELSE NULL END) AS CountAttempt,
            COUNT(distinct CASE WHEN intr.Contact  = 1 THEN intr.InteractionKey ELSE NULL END) AS CountContact,
            COUNT(distinct CASE WHEN intr.RPC  = 1 THEN intr.InteractionKey ELSE NULL END) AS CountRPC,
            COUNT(distinct CASE WHEN intr.positive  = 1 THEN intr.InteractionKey ELSE NULL END) AS CountPositive,
            COUNT(distinct CASE WHEN intr.Negative  = 1 THEN intr.InteractionKey ELSE NULL END) AS CountNegative,
            COUNT(distinct CASE WHEN intr.SettlementActivity  = 'Offer' THEN intr.InteractionKey ELSE NULL END) AS CountSettlementOffer,
            COUNT(distinct CASE WHEN intr.SettlementActivity  = 'Approved' THEN intr.InteractionKey ELSE NULL END) AS CountSettlementApproved
	FROM InteractionCategory     intr
	WHERE 1 = 1
	GROUP BY intr.PortfolioName, intr.Interactiondate, intr.CDI, intr.Exposurekey

""")
interactionsAggregated.createOrReplaceTempView("interactionsAggregated")   

In [0]:
interactionsAggregatedDebtor = spark.sql(f"""
	SELECT
			intr.PortfolioName,
            intr.Interactiondate, 
            intr.CDI, 
            MAX(CountAttempt) as CountAttempt,
            MAX(CountContact) as CountContact,
            MAX(CountRPC) as CountRPC,
            SUM(CountPositive) as CountPositive,
            SUM(CountNegative) as CountNegative,
            SUM(CountSettlementOffer) as CountSettlementOffer,
            SUM(CountSettlementApproved) as CountSettlementApproved
	FROM interactionsAggregated     intr
	WHERE 1 = 1
	GROUP BY intr.PortfolioName, intr.Interactiondate, intr.CDI

""")
interactionsAggregatedDebtor.createOrReplaceTempView("interactionsAggregatedDebtor")   

In [0]:
interactionsAggregatedPortfolio = spark.sql(f"""
	SELECT
			intr.PortfolioName,
            intr.Interactiondate, 
            SUM(CountAttempt) as CountAttempt,
            Count(DISTINCT CASE WHEN CountAttempt > 0 THEN CDI ELSE NULL END) as CountAttemptCDI,
            SUM(CountContact) as CountContact,
            SUM(CountRPC) as CountRPC,
            SUM(CountPositive) as CountPositive,
            SUM(CountNegative) as CountNegative,
            SUM(CountSettlementOffer) as CountSettlementOffer,
            SUM(CountSettlementApproved) as CountSettlementApproved
	FROM interactionsAggregatedDebtor     intr
	WHERE 1 = 1
	GROUP BY intr.PortfolioName, intr.Interactiondate

""")
interactionsAggregatedPortfolio.createOrReplaceTempView("interactionsAggregatedPortfolio")   

In [0]:
FTE = spark.sql(f"""
	SELECT
			ca.PortfolioName
   			,ca.interactiondate
			,COUNT(distinct CA.CreatedBy) AS FTE 
	FROM interactioncategory      ca 
	WHERE 1 = 1
	AND ca.ContactTypeCode IN ('1047','1048','1049','1050','1966','5157')  
    AND (ca.ExternalCompanyKey IS NULL or ca.CreatedBy <> -100 )  
	and CA.CreatedBy <> 'system'
	GROUP BY ca.PortfolioName, ca.interactiondate

""")
FTE.createOrReplaceTempView("FTE")   

In [0]:
Final = spark.sql(f"""
	SELECT
			cast('{run_date}' as date) AS BusinessDate,
            por.PortfolioName,
            cast(cal.CalendarDate as date) as CalendarDate,
            cast(FTE.FTE as integer) as FTE,
            CAST(ifnull(TT.TalkTimeDuration,0.00) AS decimal(18,4)) AS TalkTimeDuration,
            cast(ifnull(Agg.CountAttempt, 0) + ifnull(DU.DialerUnsuccessfulAttempts,0) as integer) AS TotalAttempts,
            cast(AGG.CountAttemptCDI as integer) AS UniqueCalls,
            cast(Agg.CountContact as integer) CountContact,
            cast(Agg.CountRPC as integer) CountRPC,
            cast(CASE WHEN Agg.CountRPC = 0 THEN 0 ELSE Agg.CountRPC / (Agg.CountAttempt + DU.DialerUnsuccessfulAttempts) END as decimal(18,4)) AS RpcPercent,
            cast(Agg.CountSettlementOffer as integer) AS CountSettlementOffer,
            cast(Agg.CountSettlementApproved as integer) AS CountSettlementApproved,
            cast(Agg.CountPositive as integer) AS CountPositive,
            cast(Agg.CountNegative as integer) AS CountNegative,
            CAST(CASE WHEN Agg.CountRPC = 0 THEN 0 ELSE AGG.CountPositive / Agg.CountRPC  END as decimal(18,4)) AS RpcPPT,
            CAST(CASE WHEN Agg.CountRPC = 0 THEN 0 ELSE AGG.CountNegative / Agg.CountRPC  END as decimal(18,4)) AS RpcRPT
	FROM calendar      cal 
    CROSS JOIN (SELECT DISTINCT PortfolioName FROM interaction )   por
    LEFT JOIN FTE      FTE ON FTE.PortfolioName = por.PortfolioName AND FTE.interactiondate = cal.CalendarDate
    LEFT JOIN interactionsAggregatedPortfolio AS Agg ON Agg.PortfolioName = por.PortfolioName AND Agg.interactiondate = cal.CalendarDate
    LEFT JOIN Talktime AS TT ON TT.PortfolioName = por.PortfolioName AND TT.DATEHour = cal.CalendarDate
    LEFT JOIN DialerActivities AS DU ON DU.PortfolioName = por.PortfolioName AND DU.DATEHour = cal.CalendarDate
	WHERE 1 = 1
    AND LAST_DAY(cal.CalendarDate)=LAST_DAY('{run_date}') 
    and cal.IsBusinessDay = 1
	

""")
Final.createOrReplaceTempView("Final")   

In [0]:
target_table_path_outboundcalls = f"`{env}-cdh-delivery`.investorreports.hoist_outboundcalls"

spark.sql(f"""DELETE FROM {target_table_path_outboundcalls} WHERE BusinessDate = '{run_date}' AND PortfolioName IN {PortfolioName}""")

Final = Final.withColumn("IngestionTimestamp", lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")))

Final.write \
        .format("delta") \
        .partitionBy("BusinessDate") \
        .mode("append") \
        .saveAsTable(target_table_path_outboundcalls) # in order to write in delta_lake


spark.sql(f"REPAIR TABLE {target_table_path_outboundcalls} SYNC METADATA")